# TP cœur réseau — Capacité, files et priorité

Objectif : séparer capacité de transmission, taille de file et ordre de service.
Le scénario figé offre 100 paquets/s de 1500 octets et 50 paquets/s de 200 octets
à un lien de 1 Mbit/s pendant une seconde, puis laisse la file se vider.
La charge utile offerte dépasse la capacité. Une grande file ne crée pas de débit.

Modèle synthétique déterministe : serveur unique non préemptif, durées arrondies
à la microseconde, capacité exprimée en paquets **en attente**. Une fin de service
et le départ du paquet déjà en file précèdent une arrivée au même instant.
Les arrivées simultanées sont ordonnées par identifiant. Le modèle ne contient
ni contrôle TCP, ni en-têtes, ni propagation, ni mesure d'un lien réel.

Prérequis : Python 3.10+, pandas, Jupyter ; environnement verrouillé commun du
livre : `uv run --project machine_learning_labs --locked jupyter lab`, depuis
le dossier du livre. Le CLI et ses tests utilisent seulement la bibliothèque standard.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

book_root = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "core_network_labs" / "inputs" / "scenarios.json").is_file()
    ),
    None,
)
if book_root is None:
    raise RuntimeError(
        "Ouvrir ce notebook depuis le dossier du livre ou un de ses sous-dossiers."
    )
if str(book_root) not in sys.path:
    sys.path.insert(0, str(book_root))
from core_network_labs.run_labs import build_results  # noqa: E402 - après résolution du dossier du livre

In [ ]:
results = build_results()["latency_qos.json"]["cases"]
summary = pd.DataFrame(
    [
        {"case": name, **flow}
        for name, result in results.items()
        for flow in result["flows"]
    ]
)
summary

In [ ]:
from core_network_labs.models import Packet, simulate_queue

voice = summary[summary.flow == "voice"].set_index("case")
assert (
    voice.loc["fifo_small", "p95_sojourn_ms"]
    < voice.loc["fifo_large", "p95_sojourn_ms"]
)
assert (
    voice.loc["priority_large", "p95_sojourn_ms"]
    < voice.loc["fifo_large", "p95_sojourn_ms"]
)
assert (summary.offered == summary.delivered + summary.dropped).all()
single = simulate_queue(
    [Packet("p", "voix", 0, 1000)], rate_bps=1_000_000, waiting_slots=0
)
assert single["delivered"][0]["sojourn_us"] == 8000
pd.DataFrame(results["priority_large"]["delivered"]).head(12)

## Interprétation et contre-épreuve

Le paquet isolé de 1000 octets demande exactement 8 ms de sérialisation.
Comparer le p95 **des paquets livrés** et les pertes ensemble : exclure les
paquets rejetés du délai ne les transforme pas en livraisons rapides.
La priorité n'interrompt pas le paquet en cours et ne réserve pas de place
d'admission à la voix ; des paquets prioritaires peuvent donc être rejetés.

Expliquer le compromis file courte/file longue, puis prédire l'effet d'un
trafic vocal dépassant à lui seul 1 Mbit/s. Le p95 est le quantile empirique
au rang supérieur, sans intervalle de confiance sur une population réelle.

Réinitialisation : redémarrer le noyau et exécuter toutes les cellules. Aucune
configuration système ni fichier n'est modifié. Les entrées sont
`core_network_labs/inputs/scenarios.json` ; les sorties attendues sont
`core_network_labs/results/latency_qos.json`.
